# BTS Digital Twin (NVS) — Đóng gói submission từ checkpoint đã chọn (multi-round)

Notebook này **KHÔNG train** — chỉ tải checkpoint (`gs_model/`) đã train xong của cả
7 scene (5 scene BTS + `bonsai` + `chair`, xem `pipeline/common/scenes.py`) từ Google
Drive, render `test_poses.csv` THẬT (không phải holdout) rồi đóng gói `submission.zip`
đúng format đề bài yêu cầu (xem `docs/00_MASTER_PLAN.md` mục 1).

**Bối cảnh multi-round (đọc kỹ trước khi điền `CHECKPOINT_LINKS`):** mỗi scene có thể
đã train qua 1, 2 hoặc 3 vòng (`kaggle_round1_baseline.ipynb` -> `kaggle_round2_refine.ipynb`
-> `kaggle_round3_refine.ipynb`, xem `docs/00_MASTER_PLAN.md` mục 3.2) — SỐ VÒNG CÓ THỂ
KHÁC NHAU GIỮA CÁC SCENE, vì mỗi vòng chỉ được giữ lại nếu đo Score holdout THẬT tăng
so với vòng trước (không phải cứ chạy đủ 3 vòng là tốt hơn). Việc chọn "dùng checkpoint
vòng nào cho scene nào" là QUYẾT ĐỊNH CỦA CON NGƯỜI (so sánh Score holdout đã đo được ở
từng notebook vòng trước) — notebook này KHÔNG tự quyết định, chỉ tải đúng link Google
Drive mà bạn đã dán vào `CHECKPOINT_LINKS` (Bước 5), bất kể đó là checkpoint vòng 1, 2
hay 3.

**Trước khi chạy, cần điền:**
1. Settings → Accelerator: **GPU T4 x2** (hoặc P100) → Internet: **On**.
2. `REPO_URL`/`GIT_BRANCH` ở Bước 3, `GDRIVE_URL` ở Bước 4 (dataset — cần
   `test_poses.csv` thật của từng scene).
3. `CHECKPOINT_LINKS` ở Bước 5 — dán đủ 7 link Google Drive (mỗi scene 1 link
   **THƯ MỤC `gs_model/`** của vòng đã chọn — KHÔNG phải chỉ link 1 file `point_cloud.ply`
   đơn lẻ, xem cảnh báo BUG THẬT đã sửa ở Bước 5).

**Bảo mật:** để notebook này **Private**.


## Bước 1 — Cài đặt

In [ ]:
import torch, subprocess, sys
print("CUDA available:", torch.cuda.is_available())
print("Torch:", torch.__version__, "| CUDA build:", torch.version.cuda)
# DỪNG NGAY nếu không có GPU — trước đây cell này CHỈ print() cảnh báo rồi để "Run All"
# chạy tiếp bình thường (không raise/assert gì) — dễ bị bỏ sót giữa hàng loạt dòng log
# của !pip install ngay cell sau, dẫn tới train/render "chạy" hàng chục phút/vài tiếng
# trên CPU rồi mới crash ở lần gọi .cuda() đầu tiên (torch.cuda.is_available()=False ->
# RuntimeError muộn, sau khi đã tốn thời gian tải dataset/build extension) — rất tốn
# thời gian dưới áp lực deadline (xem docs/00_MASTER_PLAN.md mục 1). Dừng NGAY tại đây
# với thông báo rõ ràng thay vì để lỗi lộ ra muộn và mù mờ hơn nhiều.
if not torch.cuda.is_available():
    raise SystemExit(
        "KHÔNG có GPU khả dụng (torch.cuda.is_available()=False). Vào Settings (góc phải) "
        "-> Accelerator -> chọn GPU T4 x2 hoặc P100 -> Save, rồi chạy lại từ đầu. "
        "KHÔNG chạy tiếp các cell sau khi chưa có GPU — train/render 3DGS cần CUDA, chạy "
        "trên CPU sẽ crash muộn (sau khi đã tốn thời gian tải dataset/cài đặt) hoặc treo "
        "vô thời hạn."
    )
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -q "pycolmap>=3.10" "scikit-image>=0.19" lpips plyfile tqdm opencv-python-headless "gdown>=6,<7"

## Bước 2 — Clone + build 3D Gaussian Splatting

Repo gốc `graphdeco-inria/gaussian-splatting` — dùng để train/render, không tự viết lại
trainer (quá nhiều chi tiết dễ sai: densification, SH coefficients...). Bước build
2 CUDA extension (`diff-gaussian-rasterization`, `simple-knn`) mất khoảng 2-5 phút.
Commit pin `54c035f7834b564019656c3e3fcc3646292f727d` (xem `docs/PORTED_KNOWLEDGE.md`
mục 2 — bản đã xác nhận có `--antialiasing`/`--depths`/`--train_test_exp`).

In [ ]:
%cd /kaggle/working
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git
%cd /kaggle/working/gaussian-splatting
!git checkout 54c035f7834b564019656c3e3fcc3646292f727d
!git submodule update --init --recursive
%cd /kaggle/working
!pip install -q ./gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install -q ./gaussian-splatting/submodules/simple-knn

import os
os.environ["GS_REPO"] = "/kaggle/working/gaussian-splatting"
print("GS_REPO =", os.environ["GS_REPO"])

## Bước 3 — Lấy code pipeline từ Git repo của bạn (khuyến nghị để **Private**)

Repo Private vẫn clone được bình thường trên Kaggle, chỉ cần xác thực bằng
**Personal Access Token (PAT)** thay vì mật khẩu. Các bước 1 lần:

1. Đẩy code lên GitHub, chọn **Private** khi tạo repo (không ai ngoài bạn xem được,
   kể cả khi bạn share notebook Kaggle này cho người khác sau này).
2. Tạo token: GitHub → **Settings → Developer settings → Personal access tokens →
   Fine-grained tokens → Generate new token**. Chọn:
   - Repository access: **Only select repositories** → chọn đúng repo vừa tạo.
   - Permissions → Contents: **Read-only** (không cần quyền gì khác).
   - Đặt ngày hết hạn (Expiration) ngắn thôi, vd 30-90 ngày — hết hạn thì tạo token mới.
3. **Copy token, dán vào Kaggle Secrets (KHÔNG dán thẳng vào code)**: trong notebook
   Kaggle, vào menu **Add-ons → Secrets → Add a new secret** → Label đặt đúng tên
   `GITHUB_TOKEN`, Value dán token vừa copy → Save. Cell bên dưới sẽ tự đọc secret
   này lúc chạy, token không hề xuất hiện trong code/notebook — kể cả nếu lỡ share
   notebook cho người khác, họ cũng không nhìn thấy được token của bạn.

`GIT_BRANCH` mặc định `"main"` — đổi nếu code cần dùng đang nằm ở 1 nhánh tích hợp
tạm thời chưa merge (vd `coordination/...`, xem `WORKLOG.md` để biết nhánh hiện hành).
Cell dưới tự dò tìm thư mục con tên `pipeline` (chứa `common/` và `scripts/`) ở bất kỳ
độ sâu nào trong repo, không cần đúng ngay gốc repo.

In [ ]:
REPO_URL = "https://github.com/ThongLuc2k3/BTS-Digital-Twin-MultiRound.git"
GIT_BRANCH = "main"  # <-- đổi nếu hạ tầng cần dùng đang nằm ở 1 nhánh tích hợp tạm thời chưa merge vào main

# GITHUB_TOKEN: ưu tiên lấy từ Kaggle Secrets (an toàn, không lộ trong code).
# Chỉ cần dán thẳng vào biến bên dưới nếu bạn KHÔNG dùng Kaggle Secrets (kém an
# toàn hơn — token sẽ nằm lộ trong notebook, đừng share notebook cho ai nếu làm vậy).
GITHUB_TOKEN = ""

try:
    from kaggle_secrets import UserSecretsClient
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
        print("Đã lấy GITHUB_TOKEN từ Kaggle Secrets.")
except Exception:
    if not GITHUB_TOKEN:
        print("Không tìm thấy Kaggle Secret 'GITHUB_TOKEN' (bỏ qua nếu repo Public, "
              "hoặc bạn đã dán token thẳng vào biến GITHUB_TOKEN ở trên).")

assert REPO_URL, "Chưa điền REPO_URL — dán link git repo chứa thư mục pipeline/ vào biến này rồi chạy lại cell."

clone_url = REPO_URL
if GITHUB_TOKEN and "github.com" in REPO_URL:
    clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")

!rm -rf /kaggle/working/_repo_clone
!git clone --depth 1 -b "{GIT_BRANCH}" "{clone_url}" /kaggle/working/_repo_clone

In [ ]:
# Tự dò thư mục "pipeline" (chứa common/ và scripts/) ở bất kỳ đâu trong repo vừa
# clone, rồi symlink về /kaggle/working/pipeline — mọi cell sau đều gọi script từ đây.
import os
import shutil
from pathlib import Path

CLONE_ROOT = Path("/kaggle/working/_repo_clone")
found = None
for dirpath, dirnames, filenames in os.walk(CLONE_ROOT):
    p = Path(dirpath)
    if p.name == "pipeline" and "common" in dirnames and "scripts" in dirnames:
        found = p
        break
if found is None and (CLONE_ROOT / "common").exists() and (CLONE_ROOT / "scripts").exists():
    found = CLONE_ROOT  # trường hợp bạn push thẳng NỘI DUNG pipeline/ làm gốc repo

if found is None:
    raise SystemExit(
        "Không tìm thấy thư mục 'pipeline' (chứa common/ và scripts/) trong repo vừa clone.\n"
        f"Nội dung clone nằm ở {CLONE_ROOT} — kiểm tra lại đã push đúng thư mục pipeline/ lên git chưa, "
        "và GIT_BRANCH ở cell trên có đúng nhánh chứa code mới nhất không."
    )

print("Tìm thấy code pipeline tại:", found)
target = Path("/kaggle/working/pipeline")
if target.is_symlink():
    target.unlink()
elif target.exists():
    shutil.rmtree(target)
os.symlink(found.resolve(), target)
print("Đã symlink -> /kaggle/working/pipeline ->", found.resolve())

Path("/kaggle/working/pipeline/work").mkdir(parents=True, exist_ok=True)

# Kiểm tra sớm: script render 03_render_test_poses.py PHẢI có mặt (thuộc phần việc
# của agent Round-1, port từ 04_render_test_poses.py của repo tiền nhiệm — xem
# docs/MILESTONE_03_submission_and_testing.md mục "tích hợp còn thiếu" nếu cell này báo lỗi).
_render_script = target / "scripts" / "03_render_test_poses.py"
_package_script = target / "scripts" / "07_package_submission.py"
assert _package_script.exists(), f"THIẾU {_package_script} — kiểm tra lại đã clone đúng nhánh/commit."
if not _render_script.exists():
    print(f"[CẢNH BÁO] Chưa thấy {_render_script} — nếu agent Round-1 chưa đẩy script này lên "
          "nhánh đang dùng, Bước 6 bên dưới sẽ lỗi 'No such file or directory'. Không phải lỗi ở "
          "notebook này, cần đợi/đổi GIT_BRANCH sang nhánh đã có script đó.")

## Bước 4 — Tải dataset từ Google Drive

Điền link chia sẻ Google Drive (chế độ "Anyone with the link") vào `GDRIVE_URL` bên
dưới — file phải là **1 file .zip** chứa `Dataset/VAI_NVS_DATA_ROUND2/<scene>/...`
(zip nguyên thư mục `Dataset` như trong repo, hoặc chỉ riêng `VAI_NVS_DATA_ROUND2`
cũng được — cell dưới tự dò tìm thư mục chứa scene ở bất kỳ độ sâu nào trong zip,
không bắt buộc đúng tên `VAI_NVS_DATA_ROUND2`, xem `docs/PORTED_KNOWLEDGE.md` mục 1).

In [ ]:
GDRIVE_URL = "https://drive.google.com/file/d/178EL7jCSVD59q19SMpeOgnOfOIC66I_t/view?usp=drive_link"

assert GDRIVE_URL, "Chưa điền GDRIVE_URL — dán link chia sẻ Google Drive (Anyone with the link) của file zip dataset vào biến này rồi chạy lại cell."

import os
os.makedirs("/kaggle/working/_dataset_raw", exist_ok=True)
!gdown "{GDRIVE_URL}" -O /kaggle/working/dataset.zip
!unzip -q -o /kaggle/working/dataset.zip -d /kaggle/working/_dataset_raw
print("Đã giải nén xong, đang dò tìm thư mục dataset ...")

In [ ]:
# Tự dò thư mục chứa các scene round 2 phẳng (HCM0421/, chair/, bonsai/...) ở bất
# kỳ đâu trong zip vừa giải nén, rồi symlink về đúng vị trí mà
# pipeline/common/scenes.py cần: /kaggle/working/Dataset/VAI_NVS_DATA_ROUND2
#
# KHÔNG bắt buộc thư mục bọc ngoài phải tên đúng "VAI_NVS_DATA_ROUND2" — chỉ cần
# TÌM ĐƯỢC 1 thư mục (kể cả chính gốc giải nén, nếu zip không có lớp bọc ngoài)
# chứa đủ NHIỀU scene mong đợi trực tiếp bên trong.
#
# Danh sách tên scene lặp lại thủ công ở đây (không import common.scenes) vì
# sys.path chưa trỏ tới pipeline/ ở bước này (việc đó làm ở cell kiểm tra ngay
# sau) — giữ đồng bộ với BTS_SCENES/GENERIC_SCENES trong pipeline/common/scenes.py
# nếu sau này thêm/bớt scene.
import os
from pathlib import Path

_expected_scene_dirs = {"HCM0421", "HCM0539", "HCM0540", "HCM0644", "HCM0674", "bonsai", "chair"}
_MIN_MATCH = 4  # đủ scene trùng khớp để tin đây đúng là thư mục dataset (tránh khớp nhầm thư mục rác)

RAW_ROOT = Path("/kaggle/working/_dataset_raw")
found = None
best_match = 0
for dirpath, dirnames, filenames in os.walk(RAW_ROOT):
    # __MACOSX/ là rác do Mac tạo khi nén zip — nó TỰ NHÂN BẢN y hệt cấu trúc thư
    # mục thật (HCM0421/, train/images/...) nhưng file bên trong chỉ là file
    # rác metadata "._<tên file>", không phải dữ liệu thật. Phải loại trừ, nếu
    # không os.walk có thể tìm trúng "__MACOSX/..." trước bản thật.
    dirnames[:] = [d for d in dirnames if d != "__MACOSX" and not d.startswith(".")]
    n_match = len(_expected_scene_dirs & set(dirnames))
    if n_match > best_match:
        best_match = n_match
        found = Path(dirpath)
    if n_match == len(_expected_scene_dirs):
        break  # khớp đủ cả 7 — dừng sớm, khỏi walk tiếp cho nhanh

if found is None or best_match < _MIN_MATCH:
    raise SystemExit(
        "Không tìm thấy thư mục nào chứa đủ scene (HCM0421, chair, bonsai...) "
        f"trong zip vừa giải nén (khớp nhiều nhất: {best_match}/7, cần >= {_MIN_MATCH}).\n"
        f"Nội dung giải nén nằm ở {RAW_ROOT} — chạy `!find {RAW_ROOT} -maxdepth 3` ở 1 cell "
        "khác để xem cấu trúc thật, đối chiếu lại với file zip đã upload lên Google Drive."
    )

print(f"Tìm thấy ({best_match}/7 scene khớp):", found)
target = Path("/kaggle/working/Dataset/VAI_NVS_DATA_ROUND2")
target.parent.mkdir(parents=True, exist_ok=True)
# PHẢI phân biệt symlink (unlink) với thư mục THẬT còn sót lại (rmtree) — nếu chỉ
# unlink() khi is_symlink() rồi bỏ qua trường hợp còn lại (bug thật tìm thấy ở
# verification pass #3: bản cũ dùng "target.unlink() if target.is_symlink() else None",
# tương đương KHÔNG LÀM GÌ nếu target là thư mục thật đã tồn tại từ trước — rồi
# "if not target.exists(): os.symlink(...)" cũng bị bỏ qua vì thư mục cũ vẫn còn đó,
# nên symlink MỚI không bao giờ được tạo, dataset MỚI tải về bị ÂM THẦM bỏ qua, notebook
# tiếp tục chạy với dữ liệu CŨ mà không báo lỗi gì) — dùng đúng cách xử lý đã verify ở
# kaggle_round1_baseline.ipynb/kaggle_round2_refine.ipynb/kaggle_round3_refine.ipynb.
if target.is_symlink():
    target.unlink()
elif target.exists():
    import shutil
    shutil.rmtree(target)
os.symlink(found.resolve(), target)
print("Đã symlink ->", target, "->", found.resolve())

# QUAN TRỌNG: code (git clone) và dataset (Google Drive) không nằm chung 1 thư mục
# gốc trên Kaggle như lúc chạy local, nên common/scenes.py KHÔNG thể tự suy ra
# đường dẫn dataset bằng "đi lên N cấp từ vị trí file code" — phải khai báo thẳng
# qua biến môi trường này (đọc bởi pipeline/common/scenes.py).
os.environ["BTS_DATASET_ROOT"] = str(target.resolve())
print("Đã set BTS_DATASET_ROOT =", os.environ["BTS_DATASET_ROOT"])

In [ ]:
# Kiểm tra lại: liệt kê đủ 7 scene + scene nào có test_poses.csv thật — dataset
# đầy đủ thì kỳ vọng csv_ok=True cho CẢ 7 scene (đây là điều kiện BẮT BUỘC để
# render/đóng gói submission, khác với kaggle_round*.ipynb chỉ cần train/sparse).
import sys
sys.path.insert(0, "/kaggle/working/pipeline")
from common.scenes import all_scenes, DATASET_ROOT

print("DATASET_ROOT =", DATASET_ROOT, "| tồn tại:", DATASET_ROOT.exists())
for s in all_scenes():
    ok_csv = s.test_poses_csv.exists()
    n_poses = 0
    if ok_csv:
        with open(s.test_poses_csv) as f:
            n_poses = sum(1 for _ in f) - 1
    print(f"{s.name:10s} {s.domain:8s} csv_ok={ok_csv} n_test_poses={n_poses:4d}")

## Bước 5 — Tải checkpoint từ Google Drive cho cả 7 scene

Điền đủ 7 link Google Drive (mỗi scene 1 link **THƯ MỤC `gs_model`** — đúng thư mục mà
`kaggle_round1_baseline.ipynb`/`kaggle_round2_refine.ipynb`/`kaggle_round3_refine.ipynb`
hướng dẫn tải nguyên vẹn lên Drive sau khi train, KHÔNG phải chỉ link 1 file
`point_cloud.ply` đơn lẻ) vào dict `CHECKPOINT_LINKS` bên dưới. Nhớ share thư mục đó ở
chế độ "Anyone with the link".

Field `"round"` trong mỗi entry **chỉ để ghi chú/theo dõi** (in ra log lúc tải, để
sau này đọc lại notebook đã Save Version biết ngay đã dùng checkpoint vòng nào cho
scene nào) — KHÔNG ảnh hưởng logic tải/render, notebook luôn tải đúng URL bạn dán ở
`"link"` bất kể ghi `"round"` là bao nhiêu. Việc chọn vòng nào là quyết định của bạn,
dựa trên Score holdout đã đo được ở từng notebook vòng train tương ứng
(`docs/00_MASTER_PLAN.md` mục 3.2 — chỉ giữ vòng nếu Score tăng đo được thật).

**LƯU Ý QUAN TRỌNG (bug thật đã tìm+sửa ở repo tiền nhiệm, xem
`docs/PORTED_KNOWLEDGE.md` mục 4):** phiên bản cell này lúc trước chỉ tải mỗi file
`point_cloud.ply` — thiếu `cfg_args` và `pipeline_train_flags.json` nằm cùng cấp trong
`gs_model/`. Thiếu 2 file đó, `03_render_test_poses.py` KHÔNG có cách nào biết lúc
train thật sự có bật `--antialiasing` hay không — nó âm thầm mặc định
`antialiasing=False` (chỉ in cảnh báo, không dừng lại), làm SAI HOÀN TOÀN ảnh render
nộp bài mà không có lỗi rõ ràng nào báo ra. Cell dưới tải NGUYÊN thư mục `gs_model`
(dùng `gdown --folder`) và tự kiểm tra đủ `cfg_args` +
`pipeline_train_flags.json` + ít nhất 1 `point_cloud.ply` trước khi cho qua — báo lỗi
rõ ràng nếu thiếu thay vì âm thầm render sai.

In [ ]:
CHECKPOINT_LINKS = {
    # scene: {"round": vòng đã CHỌN (1/2/3, chỉ để ghi chú), "link": link Drive thư mục gs_model/}
    "HCM0421": {"round": 1, "link": ""},
    "HCM0539": {"round": 1, "link": ""},
    "HCM0540": {"round": 1, "link": ""},
    "HCM0644": {"round": 1, "link": ""},
    "HCM0674": {"round": 1, "link": ""},
    "bonsai":  {"round": 1, "link": ""},
    "chair":   {"round": 1, "link": ""},
}

missing = [s for s, v in CHECKPOINT_LINKS.items() if not v.get("link")]
assert not missing, f"Chưa điền link Google Drive cho scene: {missing}"

In [ ]:
import shutil
from pathlib import Path

for scene, entry in CHECKPOINT_LINKS.items():
    link = entry["link"]
    dest_dir = Path(f"/kaggle/working/pipeline/work/{scene}")
    dest_dir.mkdir(parents=True, exist_ok=True)
    gs_model_dst = dest_dir / "gs_model"
    shutil.rmtree(gs_model_dst, ignore_errors=True)

    raw_dl_dir = Path(f"/kaggle/working/_ckpt_raw/{scene}")
    shutil.rmtree(raw_dl_dir, ignore_errors=True)
    raw_dl_dir.mkdir(parents=True, exist_ok=True)
    print(f"===== {scene} (vòng đã chọn: {entry['round']}): tải thư mục gs_model từ Drive =====")
    !gdown --folder "{link}" -O "{raw_dl_dir}"

    # gdown --folder có thể tự tạo thêm 1 lớp thư mục con trùng tên folder Drive gốc
    # (tuỳ version/cấu trúc) — tự dò lớp chứa "cfg_args" (file luôn nằm ở gốc
    # gs_model/, do train.py tự ghi) thay vì giả định cứng độ sâu.
    candidates = [p.parent for p in raw_dl_dir.rglob("cfg_args")]
    assert candidates, (
        f"{scene}: tải xong nhưng KHÔNG tìm thấy file 'cfg_args' trong {raw_dl_dir} — kiểm tra lại "
        f"link Drive có đúng là THƯ MỤC gs_model/ (không phải chỉ mỗi point_cloud.ply) và đã share "
        f"\"Anyone with the link\" chưa.")
    src_root = candidates[0]

    assert (src_root / "pipeline_train_flags.json").exists(), (
        f"{scene}: thiếu pipeline_train_flags.json trong {src_root} — thư mục gs_model tải lên Drive "
        f"phải nguyên vẹn (không tự xoá bớt file con nào), nếu không render sẽ SAI antialiasing (âm "
        f"thầm, không báo lỗi rõ — xem docstring 03_render_test_poses.py::read_pipeline_train_flags).")
    n_ply = len(list(src_root.glob("point_cloud/iteration_*/point_cloud.ply")))
    assert n_ply > 0, f"{scene}: không tìm thấy point_cloud.ply nào trong {src_root}/point_cloud/iteration_*/"

    shutil.copytree(src_root, gs_model_dst)
    shutil.rmtree(raw_dl_dir, ignore_errors=True)
    print(f"-> OK, {gs_model_dst} ({n_ply} checkpoint iteration, cfg_args + pipeline_train_flags.json có mặt)")

## Bước 6 — Render lại ảnh test THẬT cho cả 7 scene

Dùng đúng checkpoint vừa tải (iteration lớn nhất có sẵn cho mỗi scene, mặc định của
`03_render_test_poses.py`), render trên `test_poses.csv` THẬT của từng scene (khác
holdout dùng ở các notebook train — ở đây không có ảnh GT để so, chỉ sinh ảnh nộp bài).

`!lệnh` (shell magic) không tự raise exception nếu thất bại (xem
`docs/PORTED_KNOWLEDGE.md` mục 4) — cell dưới tự kiểm tra tường minh bằng
`raise SystemExit` nếu 1 scene không sinh ra thư mục `renders/` sau khi chạy, thay vì
âm thầm chạy tiếp sang scene kế và chỉ phát hiện thiếu ảnh muộn ở Bước 7.

In [ ]:
from pathlib import Path

for scene in CHECKPOINT_LINKS:
    !python /kaggle/working/pipeline/scripts/03_render_test_poses.py --scene {scene}
    renders_dir = Path(f"/kaggle/working/pipeline/work/{scene}/renders")
    if not renders_dir.exists() or not any(renders_dir.glob("*.png")):
        raise SystemExit(
            f"{scene}: 03_render_test_poses.py chạy xong nhưng KHÔNG thấy ảnh PNG nào ở {renders_dir} "
            "— kiểm tra log ở cell ngay trên (lỗi shell !python không tự dừng notebook)."
        )
    print(f"{scene}: OK, {len(list(renders_dir.glob('*.png')))} ảnh render.")

## Bước 7 — Đóng gói + kiểm tra `submission.zip`

`07_package_submission.py` tự kiểm tra đủ 7 scene / đủ ảnh / đúng kích thước trước khi
nén, và verify lại chính file zip vừa tạo (tên/đuôi file giữ nguyên `image_name`, nội
dung mã hoá lại đúng định dạng theo đuôi — xem `docs/PORTED_KNOWLEDGE.md` mục 5).

`--jpeg_quality 98`: quality cao gần lossless (PSNR ~50dB so ảnh gốc), giữ chất lượng
tối đa trong khi vẫn nén nhẹ hơn PNG nhiều — đo tham khảo ở repo tiền nhiệm cho ~304MB/
7 scene ở quality 98 (dư an toàn so hạn 350MB ở round trước đó, nhưng dataset/độ phân
giải round này có thể khác — cell dưới LUÔN tự đo lại dung lượng thật và assert lỗi
ngay nếu vượt 350MB, không tin số đo cũ).

In [ ]:
!python /kaggle/working/pipeline/scripts/07_package_submission.py \
    --out /kaggle/working/submission.zip \
    --filename_mode literal \
    --jpeg_quality 98

import os
size_mb = os.path.getsize('/kaggle/working/submission.zip') / 1e6
print(f"Dung luong zip: {size_mb:.1f} MB (gioi han 350MB)")
assert size_mb < 350, f"VUOT GIOI HAN 350MB ({size_mb:.1f}MB) — giam --jpeg_quality xuong (vd 95) roi chay lai cell nay."

## Bước 8 — Lưu kết quả

`submission.zip` đã nằm ở `/kaggle/working/` — bấm **Save Version** (góc
trên phải) để Kaggle giữ lại file này trong tab "Output" của notebook, tải về từ đó
để nộp bài. Trước khi nộp thật, đối chiếu lại checklist mục 1 + mục 10
`docs/00_MASTER_PLAN.md` (đủ 7 scene, đúng tên file, không sửa tay ảnh nào).